In [4]:
# GRUPO AXIOM - Sophie Mejía, Santiago Ibañez, Joel Martinez, Esteban Blanco

import re
import copy

class Literal:
    """ Representa un literal lógico como Hombre(x), ~Odia(Marco, Cesar) """
    def __init__(self, predicado, argumentos, negado=False):
        self.predicado = predicado
        self.argumentos = argumentos  # Lista de strings ejm ["Marco", "Cesar"]
        self.negado = negado

    def __str__(self):
        prefijo = "~" if self.negado else ""
        args = ", ".join(self.argumentos)
        return f"{prefijo}{self.predicado}({args})"

    def __eq__(self, otro):
        return (self.predicado == otro.predicado and
                self.argumentos == otro.argumentos and
                self.negado == otro.negado)

    def __hash__(self):
        return hash(str(self))

def es_variable(termino):
    """ En IA, se acostumbra que las variables empiecen con minúscula (x, y, x5) """
    if not termino:
        return False
    return termino[0].islower()
# Algoritmo de unificación
def aplicar_sustitucion_termino(termino, theta):
    """ Reemplaza una variable recursivamente si está en el diccionario de sustitución """
    if termino in theta:
        return aplicar_sustitucion_termino(theta[termino], theta)
    return termino

def sustituir(literal, theta):
    """ Aplica la sustitución theta a todos los argumentos de un literal """
    nuevos_args = [aplicar_sustitucion_termino(arg, theta) for arg in literal.argumentos]
    return Literal(literal.predicado, nuevos_args, literal.negado)

def unificar_variable(var, x, theta):
    if var in theta:
        return unificar(theta[var], x, theta)
    elif x in theta:
        return unificar(var, theta[x], theta)
    else:
        nuevo_theta = theta.copy()
        nuevo_theta[var] = x
        return nuevo_theta

def unificar(x, y, theta):
    if theta is None:
        return None
    elif x == y:
        return theta
    elif isinstance(x, str) and es_variable(x):
        return unificar_variable(x, y, theta)
    elif isinstance(y, str) and es_variable(y):
        return unificar_variable(y, x, theta)
    elif isinstance(x, list) and isinstance(y, list):
        if len(x) != len(y):
            return None
        if len(x) == 0:
            return theta
        return unificar(x[1:], y[1:], unificar(x[0], y[0], theta))
    else:
        return None


# Motor de resolucion por ref
def resolver(c1, c2):
    resolventes = []
    for l1 in c1:
        for l2 in c2:
            if l1.predicado == l2.predicado and l1.negado != l2.negado:
                theta = unificar(l1.argumentos, l2.argumentos, {})

                if theta is not None:
                    resto_c1 = [l for l in c1 if l != l1]
                    resto_c2 = [l for l in c2 if l != l2]

                    nueva_clausula_cruda = resto_c1 + resto_c2
                    nueva_clausula = [sustituir(l, theta) for l in nueva_clausula_cruda]

                    # Eliminar duplicados en la nueva cláusula
                    nueva_clausula_limpia = []
                    for nc in nueva_clausula:
                        if nc not in nueva_clausula_limpia:
                            nueva_clausula_limpia.append(nc)

                    resolventes.append((nueva_clausula_limpia, theta))
    return resolventes

def imprimir_clausula(clausula):
    if not clausula:
        return "[] (Cláusula Vacía -> CONTRADICCIÓN)"
    return " v ".join(str(l) for l in clausula)

def resolucion_FOL(base_conocimiento, consulta_negada):
    print("\n--- INICIANDO MOTOR DE RESOLUCIÓN CON UNIFICACIÓN ---")
    clausulas = copy.deepcopy(base_conocimiento)
    clausulas.append(consulta_negada)

    clausulas_procesadas_str = set()
    for c in clausulas:
        clausulas_procesadas_str.add(imprimir_clausula(c))

    iteracion = 1
    # Ponemos un límite por seguridad ante bucles infinitos
    while iteracion <= 100:
        print(f"\n--- Iteración {iteracion} ---")
        n = len(clausulas)
        nuevas_clausulas = []

        for i in range(n):
            for j in range(i + 1, n):
                resultados = resolver(clausulas[i], clausulas[j])

                for nueva_clausula, theta in resultados:
                    if len(nueva_clausula) == 0:
                        print(f"  [!] Resuelto: {imprimir_clausula(clausulas[i])}")
                        print(f"      con:      {imprimir_clausula(clausulas[j])}")
                        print(f"      Sustituciones: {theta}")
                        print("==========================================================")
                        print("=> CONTRADICCIÓN ALCANZADA (Cláusula vacía generada).")
                        print("=> EL TEOREMA ES VERDADERO.")
                        print("==========================================================")
                        return True

                    str_clausula = imprimir_clausula(nueva_clausula)
                    if str_clausula not in clausulas_procesadas_str:
                        nuevas_clausulas.append(nueva_clausula)
                        clausulas_procesadas_str.add(str_clausula)
                        print(f"Nueva cláusula deducida: {str_clausula}  (Con {theta})")

        if not nuevas_clausulas:
            print("\n=> No se pueden deducir nuevas cláusulas, por lo tanto el teorema es FALSO.")
            return False

        clausulas.extend(nuevas_clausulas)
        iteracion += 1

    print("\n=> Se alcanzó el límite de iteraciones. El teorema NO se pudo probar (o el motor entró en un bucle infinito).")
    return False

# Parsear y lectura

def parsear_literal(texto, invertir_signo=False):
    texto = texto.strip()
    match = re.match(r"(~?)\s*([A-Za-z_]\w*)\s*\(([^)]+)\)", texto)
    if not match:
        raise ValueError(f"Sintaxis incorrecta en el literal: '{texto}'")

    tiene_negacion = (match.group(1) == "~")
    predicado = match.group(2)
    argumentos = [arg.strip() for arg in match.group(3).split(",")]

    negado_final = not tiene_negacion if invertir_signo else tiene_negacion
    return Literal(predicado, argumentos, negado=negado_final)

def traducir_regla_a_fnc(regla_texto):
    if "=>" in regla_texto:
        izq_str, der_str = regla_texto.split("=>")
        literales_izq = [parsear_literal(l, invertir_signo=True) for l in izq_str.split("&")]
        literales_der = [parsear_literal(l, invertir_signo=False) for l in der_str.split("|")]
        return literales_izq + literales_der
    else:
        return [parsear_literal(l, invertir_signo=False) for l in regla_texto.split("|")]

def procesar_base_conocimiento(texto_lpo):
    bc_generada = []
    for linea in texto_lpo:
        clausula = traducir_regla_a_fnc(linea)
        bc_generada.append(clausula)
    return bc_generada


def main():
    print("="*60)
    print(" MOTOR DE INFERENCIA POR RESOLUCIÓN (Lógica de Primer Orden) ")
    print("="*60)
    print("Ingresa tu Base de Conocimiento (BC) regla por regla.")
    print("Formatos soportados:")
    print("  - Hechos:          Posee(Nono, M1)")
    print("  - Implicaciones:   Misil(x) & Posee(Nono,x) => Vende(West,x,Nono)")
    print("  - Disyunciones:    Romano(x) | Griego(x)")
    print("\nEscribe 'FIN' cuando hayas terminado de agregar reglas.\n")

    textos_input = []
    contador = 1

    # input del usuario
    while True:
        linea = input(f"Regla {contador}: ").strip()

        if linea.upper() == "FIN":
            break

        if linea:
            textos_input.append(linea)
            contador += 1

    if not textos_input:
        print("\nNo ingresaste ninguna regla. Saliendo del programa...")
        return

    print("\n" + "-"*40)
    pregunta = input("Ingresa la consulta a probar (ej. Criminal(West)): ").strip()

    if not pregunta:
        print("No ingresaste ninguna consulta. Saliendo del programa...")
        return

    try:
        # Transformamos los strings a forma normal vonjuntiva
        bc_fnc = procesar_base_conocimiento(textos_input)

        print("\n=== BASE DE CONOCIMIENTO TRANSFORMADA A FNC ===")
        for i, clausula in enumerate(bc_fnc):
            print(f" {i+1}. {imprimir_clausula(clausula)}")

        # Negamos la pregunta
        consulta_negada = [parsear_literal(pregunta, invertir_signo=True)]
        print(f"\nPregunta original: {pregunta}")
        print(f"Pregunta negada para refutación: {imprimir_clausula(consulta_negada)}")

        # 3. Lanzamos el motor de inferencia
        resolucion_FOL(bc_fnc, consulta_negada)

    except ValueError as ve:
        print(f"\n[ERROR DE SINTAXIS] Verifica tu base de conocimientos: {ve}")
    except Exception as e:
        print(f"\n[ERROR] Ocurrió un problema inesperado: {e}")

if __name__ == "__main__":
    main()

 MOTOR DE INFERENCIA POR RESOLUCIÓN (Lógica de Primer Orden) 
Ingresa tu Base de Conocimiento (BC) regla por regla.
Formatos soportados:
  - Hechos:          Posee(Nono, M1)
  - Implicaciones:   Misil(x) & Posee(Nono,x) => Vende(West,x,Nono)
  - Disyunciones:    Romano(x) | Griego(x)

Escribe 'FIN' cuando hayas terminado de agregar reglas.

Regla 1: fin

No ingresaste ninguna regla. Saliendo del programa...
